# 6장 — 검수자: verifier와 베이즈 (핵심)

웹 챕터(6장)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산합니다. 웹 실험실(VerifierLab)은
완전 해석적(난수 없음)이라, 여기서도 공식 그대로 재현하고 assert 로 자가검증해요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch6')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 공식: 통과 더미에서 정답의 비율

기저율 $p$(답이 애초에 맞을 확률), 검수자의 민감도 $s$(맞는 답을 통과), 특이도 $t$(틀린 답을 거름)일 때

$$P(\text{정답}\mid\text{통과}) = \frac{p \cdot s}{p \cdot s + (1-p)(1-t)}$$

분자 = 통과한 정답, 분모 = 통과 더미 전체(통과한 정답 + 새는 오답). 3부 4장 스팸 필터의 그 베이즈입니다.

In [ ]:
def posterior(p, s, t):
    """검수 통과 후 정답일 확률 — 베이즈 사후확률 (웹 VerifierLab 과 동일 공식)."""
    return (p * s) / (p * s + (1 - p) * (1 - t))

# 기본 설정: 워밍업 일꾼(기저율 35%) + 검수자(민감도 90%·특이도 80%)
base = posterior(0.35, 0.9, 0.8)
print(f'검수 전 35% → 통과 후 {base*100:.1f}%')
assert abs(base * 100 - 70.8) < 0.05

# 답안 100개 그리드 (웹 ② 와 동일한 반올림 정수)
correct, wrong = 35, 65
cp, wp = round(correct * 0.9), round(wrong * 0.2)
print(f'통과 더미: 정답 {cp} + 새는 오답 {wp} → 비율 {cp/(cp+wp)*100:.1f}%')
assert (cp, correct - cp, wp, wrong - wp) == (32, 3, 13, 52)

# 기저율의 역설: 어려운 과업(5%)은 통과해도 —
hard = posterior(0.05, 0.9, 0.8)
print(f'기저율 5% → 통과 후 {hard*100:.1f}% (여전히 못 믿음!)')
assert abs(hard * 100 - 19.1) < 0.05

# StepQuiz 검산: 기저율 50%
mid = posterior(0.5, 0.9, 0.8)
assert abs(0.5 * 0.9 - 0.45) < 1e-12 and abs(0.45 + 0.5 * 0.2 - 0.55) < 1e-12
assert abs(mid * 100 - 81.8) < 0.05

# 반복 검수(독립!): 사후확률이 다음 검수의 기저율로 — 베이즈 갱신의 연쇄
twice = posterior(base, 0.9, 0.8)
thrice = posterior(twice, 0.9, 0.8)
print(f'반복 검수: 35% → {base*100:.1f}% → {twice*100:.1f}% → {thrice*100:.1f}%')
assert abs(twice * 100 - 91.6) < 0.05 and abs(thrice * 100 - 98.0) < 0.05

# 민감도 vs 특이도 — 통과의 신뢰는 특이도가 지배한다
s_up = posterior(0.35, 0.95, 0.8)
t_up = posterior(0.35, 0.9, 0.95)
print(f'민감도 90→95%: {s_up*100:.1f}%  vs  특이도 80→95%: {t_up*100:.1f}%')
assert abs(s_up * 100 - 71.9) < 0.05 and abs(t_up * 100 - 90.6) < 0.05
print('✅ 웹 실험실·본문 수치 자가검증 통과')

## 2. 기저율 스윕: 같은 검수자, 다른 바탕

검수자의 실력(민감도·특이도)을 고정하고 기저율만 훑으면, "통과 도장의 무게"가 바탕에 얼마나
좌우되는지 곡선 하나로 보입니다. 점선(y = x)은 검수가 아무 정보도 못 주는 경우. 곡선이
점선에서 멀수록 검수의 값어치가 큽니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

prior = np.linspace(0.01, 0.95, 200)
fig, ax = plt.subplots(figsize=(7.5, 5))
for (s, t), color in [((0.9, 0.8), '#6366f1'), ((0.9, 0.95), '#10b981'), ((0.99, 0.99), '#f59e0b')]:
    ax.plot(prior * 100, posterior(prior, s, t) * 100, color=color,
            label=f'민감도 {s:.0%} · 특이도 {t:.0%}')
ax.plot([0, 100], [0, 100], ls='--', lw=1, color='#94a3b8', label='검수 없음 (y = x)')
for x0, note in [(35, '워밍업 일꾼'), (5, '어려운 과업')]:
    y0 = posterior(x0 / 100, 0.9, 0.8) * 100
    ax.scatter([x0], [y0], color='#6366f1', zorder=5)
    ax.annotate(f'{note}\n{x0}%→{y0:.1f}%', (x0, y0), textcoords='offset points',
                xytext=(10, -18), fontsize=9, color='#334155')
ax.set_xlabel('기저율 p (검수 전, %)')
ax.set_ylabel('통과 후 정답 확률 (%)')
ax.set_title('검수 도장의 무게는 바탕(기저율)과 검수자(특이도)가 정한다')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 오늘의 발견

- **통과 후의 믿음 = 통과 더미에서 정답의 비율**: 기저율 35%가 검수 한 번에 70.8%로 도약.
- **기저율의 역설**: 바탕이 5%면 좋은 검수자를 통과해도 19.1%. 새는 오답이 더미를 지배.
- 통과의 신뢰는 **특이도**가 지배(80→95%에서 70.8→90.6%), 반복 검수는 **독립일 때만** 연쇄 상승(→91.6→98.0%).
- 함정 둘: 자기 검수(상관, 같은 맹점), 굿하트의 법칙(검수 점수가 목표가 되면 검수만 통과하는 답이 진화).

이로써 되살리는 세 가지 길(다시 뽑기·다수결·검수)이 완성. 다음 장(7장 생각의 길이)에서 가격표를 폅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**